In [ ]:
# =============================================================================
# NOTEBOOK A — PREPROCESSING  (MobileCLIP2-S0 + MobileNetV3-Small Edition)
# Single-Phase: Download Once → RAM → Shared Memory → Parallel GPU Encoding
#
# Feature extractors:
#   1. MobileCLIP2-S0  (open_clip)  → 512-d semantic embedding
#   2. MobileNetV3-Small (torchvision, pretrained ImageNet)
#      Multi-layer hooks → GAP → concat → 184-d low/mid-level features
#      Layers tapped:
#        features[1]  : 16-ch  — edges, color gradients
#        features[3]  : 24-ch  — early textures, color regions
#        features[6]  : 48-ch  — mid-level form / shape primitives
#        features[10] : 96-ch  — complex shapes (pre-high-level)
#      Rationale: these layers precede the aggressive channel expansion at
#      features[11] (576-ch) and retain the colorful, spatially-structured
#      information that CLIP's contrastive objective discards.
# =============================================================================

# ── CELL 0 : UNBUFFERED OUTPUT (must be first) ────────────────────────────────
import os, sys
os.environ["PYTHONUNBUFFERED"] = "1"

def log(msg):
    print(msg, flush=True)

log("=== Notebook A starting ===")

# ── CELL 1 : INSTALL ──────────────────────────────────────────────────────────
import subprocess
log("Installing packages …")
subprocess.run(["pip", "install", "-q",
    "datasets", "webdataset", "transformers",
    "nilearn", "nibabel", "scipy", "huggingface_hub",
    "open-clip-torch",
    "torchvision"],          # ← added for MobileNetV3-Small
    check=True)
log("Packages installed.")

# ── CELL 2 : IMPORTS ──────────────────────────────────────────────────────────
import io, re, json, pickle, random, warnings, time
from pathlib import Path
from multiprocessing.shared_memory import SharedMemory

import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
import torchvision.models as tv_models

import nilearn.datasets as nl_datasets
import nilearn.image as nl_image
from datasets import load_dataset
import open_clip

warnings.filterwarnings("ignore")
log(f"PyTorch    : {torch.__version__}")
log(f"CUDA       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        log(f"  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB VRAM")

# ── HF TOKEN ──────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf_token
    import huggingface_hub
    huggingface_hub.login(token=_hf_token, add_to_git_credential=False)
    log("HF token loaded ✓")
except Exception as _e:
    log(f"HF token not loaded ({_e}) — continuing unauthenticated")

# ── CELL 3 : CONFIG ───────────────────────────────────────────────────────────
# MobileNetV3-Small hook layers and their output channel counts.
# GAP over spatial dims → concat → 16+24+48+96 = 184-d vector.
_MNV3_HOOK_LAYERS   = [1, 3, 6, 10]
_MNV3_LAYER_CHANNELS = {1: 16, 3: 24, 6: 48, 10: 96}
_MNV3_DIM = sum(_MNV3_LAYER_CHANNELS[l] for l in _MNV3_HOOK_LAYERS)  # 184

CFG = {
    "hf_dataset_id":          "PPWangyc/WAVE-BOLD5000",
    "hf_token":               None,
    "tr_peak_indices":        [1, 2],
    "bold_dim":               1024,
    "yeo_visual":             1,
    "yeo_somatomotor":        2,
    "yeo_dorsal_attention":   3,
    "yeo_ventral_attention":  4,
    "yeo_limbic":             5,
    "yeo_frontoparietal":     6,
    "yeo_dmn":                7,
    # ── MobileCLIP ────────────────────────────────────────────────────────────
    "clip_model_id":          "MobileCLIP2-S0",
    "clip_dim":               512,
    "clip_batch_size":        128,
    # ── MobileNetV3-Small ─────────────────────────────────────────────────────
    "mobilenet_hook_layers":  _MNV3_HOOK_LAYERS,    # [1, 3, 6, 10]
    "mobilenet_dim":          _MNV3_DIM,             # 184
    "mobilenet_batch_size":   256,                   # larger batch fine — no full forward
    # ── misc ──────────────────────────────────────────────────────────────────
    "output_dir":             "/kaggle/working/preprocessed",
    "n_test_hold_out":        30,
    "test_seed":              42,
    "subjects":               ["CSI1", "CSI2", "CSI3", "CSI4"],
    "device":                 "cuda" if torch.cuda.is_available() else "cpu",
    "difumo_resolution_mm":   3,
}

SUBJECTS = CFG["subjects"]
os.makedirs(CFG["output_dir"], exist_ok=True)
log(f"Output dir       : {CFG['output_dir']}")
log(f"Device           : {CFG['device']}")
log(f"MobileNetV3 dim  : {CFG['mobilenet_dim']}  "
    f"(layers {CFG['mobilenet_hook_layers']})")

# ── CELL 4 : YEO-7 ↔ DIFUMO-1024 MAPPING ─────────────────────────────────────
CACHE_PATH = os.path.join(CFG["output_dir"], "yeo7_difumo1024_assignment.npy")

def compute_yeo7_difumo_mapping(resolution_mm=3):
    log("Fetching DiFuMo 1024 atlas …")
    difumo = nl_datasets.fetch_atlas_difumo(dimension=1024, resolution_mm=resolution_mm)
    log("Fetching Yeo-7 parcellation …")
    yeo = nl_datasets.fetch_atlas_yeo_2011()

    difumo_img = nl_image.load_img(difumo.maps)
    yeo_img    = nl_image.load_img(yeo.maps)

    log("Resampling Yeo-7 to DiFuMo space …")
    ref_img  = nl_image.index_img(difumo_img, 0)
    yeo_res  = nl_image.resample_to_img(yeo_img, ref_img, interpolation="nearest")
    yeo_data = yeo_res.get_fdata().astype(np.int8)

    log("Loading DiFuMo maps into RAM (~1 GB) …")
    difumo_data = difumo_img.get_fdata(dtype=np.float32)

    log("Computing per-component Yeo-7 assignment …")
    n_components = difumo_data.shape[-1]
    assignment   = np.zeros(n_components, dtype=np.int8)

    for i in range(n_components):
        if i % 100 == 0:
            log(f"  DiFuMo → Yeo-7: {i}/{n_components}")
        comp = difumo_data[:, :, :, i]
        if comp.max() == 0:
            continue
        threshold  = np.percentile(comp[comp > 0], 80)
        footprint  = comp > threshold
        yeo_labels = yeo_data[footprint]
        yeo_labels = yeo_labels[yeo_labels > 0]
        if len(yeo_labels) == 0:
            continue
        counts        = np.bincount(yeo_labels.astype(np.int64), minlength=8)
        assignment[i] = int(counts[1:].argmax()) + 1

    del difumo_data
    return assignment

if os.path.exists(CACHE_PATH):
    yeo_assignment = np.load(CACHE_PATH)
    log(f"Loaded cached Yeo-7 assignment from {CACHE_PATH}")
else:
    yeo_assignment = compute_yeo7_difumo_mapping(CFG["difumo_resolution_mm"])
    np.save(CACHE_PATH, yeo_assignment)
    log(f"Saved Yeo-7 assignment to {CACHE_PATH}")

LIMBIC_MASK = (yeo_assignment == CFG["yeo_limbic"])
DMN_MASK    = (yeo_assignment == CFG["yeo_dmn"])
VISUAL_MASK = (yeo_assignment == CFG["yeo_visual"])
FPN_MASK    = (yeo_assignment == CFG["yeo_frontoparietal"])

log("\nYeo-7 parcel counts (out of 1024):")
for name, lbl in [("Visual",1),("Somatomotor",2),("DorsAttn",3),
                  ("VentAttn",4),("Limbic",5),("FPN",6),("DMN",7)]:
    log(f"  {name:<14}: {int((yeo_assignment == lbl).sum())}")

# ── CELL 5a : VERIFY MOBILECLIP2-S0 ──────────────────────────────────────────
log(f"\nVerifying {CFG['clip_model_id']} in open_clip registry …")
_available_tags = open_clip.list_pretrained_tags_by_model(CFG["clip_model_id"])
if not _available_tags:
    _mobile_models = [m for m in open_clip.list_models() if "Mobile" in m]
    raise RuntimeError(
        f"'{CFG['clip_model_id']}' has no pretrained tags.\n"
        f"Available MobileCLIP models: {_mobile_models}"
    )
_pretrained_tag = _available_tags[0]
log(f"  MobileCLIP model : {CFG['clip_model_id']}")
log(f"  MobileCLIP tag   : {_pretrained_tag}")
log(f"  MobileCLIP dim   : {CFG['clip_dim']}")

# ── CELL 5b : VERIFY MOBILENETV3-SMALL ───────────────────────────────────────
log(f"\nVerifying MobileNetV3-Small (torchvision) …")
_mnv3_test = tv_models.mobilenet_v3_small(
    weights=tv_models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
)
_n_features_layers = len(_mnv3_test.features)
log(f"  torchvision MobileNetV3-Small loaded ✓")
log(f"  features layers  : {_n_features_layers}  "
    f"(hooking indices {CFG['mobilenet_hook_layers']})")
log(f"  output dim       : {CFG['mobilenet_dim']}  "
    f"(16+24+48+96 after GAP)")

# Confirm channel counts by a dummy forward to catch any torchvision version skew
_hooks_test = {}
_handles_test = []
def _make_test_hook(idx):
    def _h(module, inp, out):
        _hooks_test[idx] = out.shape[1]
    return _h

for _l in CFG["mobilenet_hook_layers"]:
    _handles_test.append(
        _mnv3_test.features[_l].register_forward_hook(_make_test_hook(_l))
    )
_mnv3_test.eval()
with torch.no_grad():
    _mnv3_test(torch.zeros(1, 3, 224, 224))
for _h in _handles_test:
    _h.remove()

_real_dim = sum(_hooks_test[l] for l in CFG["mobilenet_hook_layers"])
log(f"  Confirmed channel counts: { {l: _hooks_test[l] for l in CFG['mobilenet_hook_layers']} }")
log(f"  Confirmed total dim      : {_real_dim}")
if _real_dim != CFG["mobilenet_dim"]:
    log(f"  WARNING: expected {CFG['mobilenet_dim']} but got {_real_dim} — updating CFG")
    CFG["mobilenet_dim"] = _real_dim
del _mnv3_test, _hooks_test, _handles_test

# ── CELL 6 : HELPERS ──────────────────────────────────────────────────────────
def _decode_bold(raw):
    if raw is None:
        return None
    if isinstance(raw, np.ndarray):
        return raw.astype(np.float32)
    if isinstance(raw, (list, tuple)):
        return np.asarray(raw, dtype=np.float32)
    if isinstance(raw, (bytes, bytearray)):
        try:
            return np.asarray(pickle.loads(raw), dtype=np.float32)
        except Exception:
            return None
    return None

def _to_img_bytes(raw) -> bytes:
    if isinstance(raw, (bytes, bytearray)):
        return bytes(raw)
    if isinstance(raw, Image.Image):
        buf = io.BytesIO()
        raw.convert("RGB").save(buf, format="PNG")
        return buf.getvalue()
    return None

def _decode_config(raw):
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, (bytes, bytearray)):
        raw = raw.decode("utf-8")
    if isinstance(raw, str):
        return json.loads(raw)
    return {}

# ── CELL 7 : DOWNLOAD HF DATASET → RAM ────────────────────────────────────────
log("\n" + "="*70)
log("  STEP 1 — Download HF dataset → RAM")
log("="*70)
log(f"  Subjects   : {SUBJECTS}")
log(f"  Note: HF download progress goes to stderr — normal in commit mode")
log("="*70)

hf_kwargs = {}
if CFG.get("hf_token"):
    hf_kwargs["token"] = CFG["hf_token"]

log("\n  Calling load_dataset() — downloading parquet files ...")
log("  HF progress goes to stderr; heartbeat below proves it is running.")
t_dl      = time.perf_counter()
_ds_result = [None]
_ds_error  = [None]

def _download_thread():
    try:
        _ds_result[0] = load_dataset(CFG["hf_dataset_id"], **hf_kwargs)
    except Exception as e:
        _ds_error[0] = e

import threading
_t = threading.Thread(target=_download_thread, daemon=True)
_t.start()
while _t.is_alive():
    _t.join(timeout=15)
    if _t.is_alive():
        log(f"  ... still downloading ({(time.perf_counter()-t_dl)/60:.1f} min elapsed)")
if _ds_error[0] is not None:
    raise RuntimeError(f"load_dataset failed: {_ds_error[0]}") from _ds_error[0]
raw_ds = _ds_result[0]
log(f"  load_dataset() complete in {(time.perf_counter()-t_dl)/60:.1f} min")

splits    = list(raw_ds.keys())
TOTAL_EST = sum(len(raw_ds[s]) for s in splits)
log(f"  Splits : {splits}")
log(f"  Total rows : {TOTAL_EST:,}\n")

_raw_storage = {s: {"bold": [], "img_bytes": [], "img_names": [],
                    "labels": [], "is_rep": []} for s in SUBJECTS}

n_parsed, n_skip, n_no_subj = 0, 0, 0
_ema_rate, _ema_alpha = None, 0.1
t0 = t_last = time.perf_counter()
n_last = 0
LOG_INTERVAL_S = 15

def _log_progress(force=False):
    global t_last, n_last, _ema_rate
    now = time.perf_counter()
    if not force and (now - t_last) < LOG_INTERVAL_S:
        return
    dt   = now - t_last
    inst = (n_parsed - n_last) / max(dt, 1e-6)
    _ema_rate = inst if _ema_rate is None else (
        _ema_alpha * inst + (1 - _ema_alpha) * _ema_rate
    )
    eta_s   = (TOTAL_EST - n_parsed) / max(_ema_rate, 1e-6)
    eta_str = f"{int(eta_s//3600)}h {int((eta_s%3600)//60)}m {int(eta_s%60)}s"
    pct     = 100 * n_parsed / max(TOTAL_EST, 1)
    subj_str = "  ".join(
        f"{s}={len(_raw_storage[s]['bold'])}(rep={sum(_raw_storage[s]['is_rep'])})"
        for s in SUBJECTS
    )
    log(f"  [{(now-t0)/60:5.1f} min | {n_parsed:>6}/{TOTAL_EST} ({pct:4.1f}%) | "
        f"~{_ema_rate:5.1f} samp/s | ETA {eta_str} | skip={n_skip}]")
    log(f"    {subj_str}")
    t_last, n_last = now, n_parsed

for split in splits:
    log(f"  ── Iterating split: {split} ──────────────────────────────────────")
    n_split = 0
    for item in raw_ds[split]:
        try:
            key = item.get("__key__", "")
            m   = re.search(r"(CSI[1-4])", key, re.IGNORECASE)
            if not m:
                n_no_subj += 1
                continue
            subj = m.group(1).upper()
            if subj not in SUBJECTS:
                continue

            bold_5tr = _decode_bold(item.get("bold.pyd"))
            if bold_5tr is None or bold_5tr.shape != (5, CFG["bold_dim"]):
                n_skip += 1
                continue

            img_bytes = _to_img_bytes(item.get("stimuli.png"))
            if img_bytes is None:
                n_skip += 1
                continue

            cfg_d    = _decode_config(item.get("config.json", "{}"))
            img_name = cfg_d.get("img_name", key)
            label    = cfg_d.get("label", "unknown")
            is_rep   = bool(cfg_d.get("rep", False))

            _raw_storage[subj]["bold"].append(bold_5tr)
            _raw_storage[subj]["img_bytes"].append(img_bytes)
            _raw_storage[subj]["img_names"].append(img_name)
            _raw_storage[subj]["labels"].append(label)
            _raw_storage[subj]["is_rep"].append(is_rep)

            n_parsed += 1
            n_split  += 1
            _log_progress()

        except Exception:
            n_skip += 1

    log(f"  Split '{split}' done: {n_split} samples")

_log_progress(force=True)
total_s = time.perf_counter() - t0
log(f"\n  Iteration complete:")
log(f"    Accepted  : {n_parsed:,}")
log(f"    Skipped   : {n_skip}")
log(f"    No-subj   : {n_no_subj}")
log(f"    Elapsed   : {total_s/60:.1f} min  ({n_parsed/max(total_s,1e-6):.1f} samp/s avg)")

log("\n  Consolidating arrays …")
storage = {}
for subj in SUBJECTS:
    rs = _raw_storage[subj]
    n  = len(rs["bold"])
    if n == 0:
        log(f"  WARNING: {subj} has 0 samples")
        continue
    bold_arr = np.stack(rs["bold"]).astype(np.float32)
    n_rep    = sum(rs["is_rep"])
    bold_mb  = bold_arr.nbytes / 1e6
    img_mb   = sum(len(b) for b in rs["img_bytes"]) / 1e6
    storage[subj] = {
        "bold":      bold_arr,
        "img_bytes": rs["img_bytes"],
        "img_names": rs["img_names"],
        "labels":    rs["labels"],
        "is_rep":    rs["is_rep"],
    }
    log(f"  {subj}: N={n}  rep={n_rep}  BOLD={bold_mb:.0f} MB  images={img_mb:.0f} MB")

del _raw_storage

# ── CELL 8 : MOVE DATA INTO POSIX SHARED MEMORY ───────────────────────────────
log(f"\n{'='*70}")
log("  Moving data to POSIX shared memory …")
log("="*70)

shm_handles = {}
shm_meta    = {}

for subj, sd in storage.items():
    n       = len(sd["bold"])
    bold_np = sd["bold"]

    shm_bold = SharedMemory(create=True, size=bold_np.nbytes)
    np.ndarray(bold_np.shape, dtype=bold_np.dtype, buffer=shm_bold.buf)[:] = bold_np

    img_list = sd["img_bytes"]
    offsets  = np.zeros(n + 1, dtype=np.int64)
    for i, b in enumerate(img_list):
        offsets[i + 1] = offsets[i] + len(b)
    total_img_bytes = int(offsets[-1])

    flat_imgs = np.empty(total_img_bytes, dtype=np.uint8)
    for i, b in enumerate(img_list):
        flat_imgs[offsets[i]:offsets[i + 1]] = np.frombuffer(b, dtype=np.uint8)

    shm_imgs = SharedMemory(create=True, size=flat_imgs.nbytes)
    np.ndarray(flat_imgs.shape, dtype=flat_imgs.dtype, buffer=shm_imgs.buf)[:] = flat_imgs

    shm_offs = SharedMemory(create=True, size=offsets.nbytes)
    np.ndarray(offsets.shape, dtype=offsets.dtype, buffer=shm_offs.buf)[:] = offsets

    shm_handles[subj] = (shm_bold, shm_imgs, shm_offs)
    shm_meta[subj] = {
        "bold_shm_name": shm_bold.name,
        "bold_shape":    bold_np.shape,
        "imgs_shm_name": shm_imgs.name,
        "imgs_size":     total_img_bytes,
        "offs_shm_name": shm_offs.name,
        "n":             n,
        "img_names":     sd["img_names"],
        "labels":        sd["labels"],
        "is_rep":        sd["is_rep"],
    }
    total_shm_mb = (shm_bold.size + shm_imgs.size + shm_offs.size) / 1e6
    log(f"  {subj}: {n} samples → {total_shm_mb:.0f} MB in shared memory")

del storage
log("  All subjects in shared memory. Python-side copies released.")
log("="*70)

# ── CELL 9 : WRITE WORKER MODULE ──────────────────────────────────────────────
#
# Each worker encodes a subject subset with BOTH models in a single image pass:
#   • MobileCLIP2-S0  → clip_embeds   (N, 512)
#   • MobileNetV3-Small multi-layer  → mobilenet_feats  (N, 184)
#
# MobileNetV3 preprocessing follows the torchvision default:
#   Resize(256) → CenterCrop(224) → ToTensor → Normalize(ImageNet)
# Forward hooks on features[1,3,6,10] collect intermediate activations;
# global average pooling collapses spatial dims; concat produces 184-d.
#
_WORKER_SRC = '''
import os, io, sys, time
import numpy as np
from PIL import Image
from multiprocessing.shared_memory import SharedMemory
import torch
import torchvision.transforms as T
import torchvision.models as tv_models
import open_clip

os.environ["PYTHONUNBUFFERED"] = "1"

def log(msg):
    print(msg, flush=True)

def _gpu_mem(device):
    try:
        idx   = int(str(device).split(":")[-1])
        used  = torch.cuda.memory_allocated(idx) / 1e9
        total = torch.cuda.get_device_properties(idx).total_memory / 1e9
        return f"{used:.2f}/{total:.1f} GB"
    except Exception:
        return "N/A"


# ─────────────────────────────────────────────────────────────────────────────
# MobileNetV3-Small feature extractor with forward hooks
# ─────────────────────────────────────────────────────────────────────────────
class MNV3FeatureExtractor(torch.nn.Module):
    """
    Wraps MobileNetV3-Small and exposes multi-layer GAP features.

    For a batch of images (B, 3, 224, 224) → returns (B, total_dim) where
    total_dim = sum of channels at each hooked layer.

    Hooked layers and their spatial-channel roles:
        features[1]  : 16-ch  — first inverted residual; edges, color boundaries
        features[3]  : 24-ch  — early texture and color region responses
        features[6]  : 48-ch  — mid-level form and shape primitives (with SE)
        features[10] : 96-ch  — complex shape encodings before high-level collapse
    """
    def __init__(self, hook_layers, device):
        super().__init__()
        self.hook_layers = hook_layers
        self.device      = device
        self._activations = {}
        self._handles    = []

        weights = tv_models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        base    = tv_models.mobilenet_v3_small(weights=weights)
        # We only need features; drop classifier and avgpool to save memory.
        self.features = base.features
        del base

        self._gap = torch.nn.AdaptiveAvgPool2d(1)

        for layer_idx in hook_layers:
            handle = self.features[layer_idx].register_forward_hook(
                self._make_hook(layer_idx)
            )
            self._handles.append(handle)

    def _make_hook(self, idx):
        def _hook(module, inp, out):
            # out : (B, C, H, W) → GAP → (B, C)
            self._activations[idx] = self._gap(out).squeeze(-1).squeeze(-1)
        return _hook

    def forward(self, x):
        self._activations.clear()
        # Drive the forward pass up to and through the deepest hooked layer.
        # We stop at max(hook_layers)+1 to avoid wasting compute on later layers.
        stop = max(self.hook_layers) + 1
        with torch.no_grad():
            for i, layer in enumerate(self.features):
                x = layer(x)
                if i >= stop:
                    break
        # Concatenate hooked activations in layer order → deterministic dim layout
        parts = [self._activations[l] for l in self.hook_layers]
        return torch.cat(parts, dim=1)   # (B, total_dim)

    def remove_hooks(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()


# ─────────────────────────────────────────────────────────────────────────────
def encode_worker(rank, cfg, shm_meta):
    device      = f"cuda:{rank}"
    all_subjects = cfg["subjects"]
    mid          = len(all_subjects) // 2
    subjects     = all_subjects[:mid] if rank == 0 else all_subjects[mid:]
    out_dir      = cfg["output_dir"]
    clip_bs      = cfg["clip_batch_size"]
    mnv3_bs      = cfg["mobilenet_batch_size"]
    hook_layers  = cfg["mobilenet_hook_layers"]

    log(f"[GPU {rank}] Worker starting  device={device}  subjects={subjects}")

    # ── Load MobileCLIP ──────────────────────────────────────────────────────
    tags = open_clip.list_pretrained_tags_by_model(cfg["clip_model_id"])
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        cfg["clip_model_id"], pretrained=tags[0]
    )
    clip_model = clip_model.to(device).eval()
    log(f"[GPU {rank}] MobileCLIP loaded  GPU={_gpu_mem(device)}")

    if hasattr(torch, "compile"):
        try:
            clip_model.visual = torch.compile(
                clip_model.visual, mode="reduce-overhead"
            )
            log(f"[GPU {rank}] MobileCLIP torch.compile ✓")
        except Exception as ce:
            log(f"[GPU {rank}] torch.compile skipped: {ce}")

    # ── Load MobileNetV3-Small ───────────────────────────────────────────────
    mnv3_model = MNV3FeatureExtractor(hook_layers, device).to(device).eval()
    # Standard ImageNet normalization used by torchvision MobileNetV3 weights
    mnv3_preprocess = T.Compose([
        T.Resize(256, interpolation=T.InterpolationMode.BILINEAR),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])
    log(f"[GPU {rank}] MobileNetV3-Small loaded  GPU={_gpu_mem(device)}")
    log(f"[GPU {rank}]   hook layers : {hook_layers}")
    log(f"[GPU {rank}]   output dim  : {cfg['mobilenet_dim']}")

    # ── Warm-up both models ───────────────────────────────────────────────────
    _dummy = torch.zeros(1, 3, 256, 256, device=device)
    with torch.no_grad():
        clip_model.encode_image(_dummy)
    _dummy224 = torch.zeros(1, 3, 224, 224, device=device)
    with torch.no_grad():
        mnv3_model(_dummy224)
    del _dummy, _dummy224
    torch.cuda.synchronize(device)
    log(f"[GPU {rank}] Warm-up done  GPU={_gpu_mem(device)}")

    # ── Per-subject encoding ──────────────────────────────────────────────────
    def _encode_clip(pil_list):
        imgs = torch.stack([clip_preprocess(im) for im in pil_list]).to(device)
        with torch.no_grad():
            e = clip_model.encode_image(imgs)
            e = e / e.norm(dim=-1, keepdim=True)
        return e.cpu().numpy().astype(np.float32)

    def _encode_mnv3(pil_list):
        imgs = torch.stack([mnv3_preprocess(im) for im in pil_list]).to(device)
        with torch.no_grad():
            f = mnv3_model(imgs)
        return f.cpu().numpy().astype(np.float32)

    for subj in subjects:
        if subj not in shm_meta:
            log(f"[GPU {rank}] {subj} not in shm_meta — skipping")
            continue

        meta = shm_meta[subj]
        n    = meta["n"]
        log(f"[GPU {rank}] Starting {subj}  N={n}")

        shm_bold = SharedMemory(name=meta["bold_shm_name"])
        shm_imgs = SharedMemory(name=meta["imgs_shm_name"])
        shm_offs = SharedMemory(name=meta["offs_shm_name"])

        bold_arr = np.ndarray(meta["bold_shape"],  dtype=np.float32, buffer=shm_bold.buf)
        flat_img = np.ndarray((meta["imgs_size"],), dtype=np.uint8,  buffer=shm_imgs.buf)
        offsets  = np.ndarray((n + 1,),            dtype=np.int64,   buffer=shm_offs.buf)

        # Use the larger of the two batch sizes to iterate once; both encoders
        # process the same PIL images per iteration (avoids double image decode).
        batch_size  = max(clip_bs, mnv3_bs)
        n_batches   = (n + batch_size - 1) // batch_size
        clip_parts  = []
        mnv3_parts  = []
        t0     = time.perf_counter()
        t_last = t0
        n_last = 0
        ema_rate = None
        LOG_INTERVAL_S = 15

        for bi, start in enumerate(range(0, n, batch_size)):
            end = min(start + batch_size, n)
            pil_imgs = [
                Image.open(io.BytesIO(
                    flat_img[offsets[i]:offsets[i + 1]].tobytes()
                )).convert("RGB")
                for i in range(start, end)
            ]

            # ── CLIP (may use a sub-slice if clip_bs < batch_size) ───────────
            clip_batch_feats = []
            for sub_s in range(0, len(pil_imgs), clip_bs):
                clip_batch_feats.append(
                    _encode_clip(pil_imgs[sub_s: sub_s + clip_bs])
                )
            clip_parts.append(np.vstack(clip_batch_feats))

            # ── MobileNetV3 (may use a sub-slice if mnv3_bs < batch_size) ────
            mnv3_batch_feats = []
            for sub_s in range(0, len(pil_imgs), mnv3_bs):
                mnv3_batch_feats.append(
                    _encode_mnv3(pil_imgs[sub_s: sub_s + mnv3_bs])
                )
            mnv3_parts.append(np.vstack(mnv3_batch_feats))

            now = time.perf_counter()
            if (now - t_last) >= LOG_INTERVAL_S:
                dt       = now - t_last
                inst_r   = (end - n_last) / max(dt, 1e-6)
                ema_rate = inst_r if ema_rate is None else (
                    0.2 * inst_r + 0.8 * ema_rate
                )
                eta_s = (n - end) / max(ema_rate, 1e-6)
                pct   = 100 * end / n
                log(f"[GPU {rank}] {subj} "
                    f"batch {bi+1}/{n_batches} | "
                    f"{end}/{n} ({pct:.1f}%) | "
                    f"~{ema_rate:.1f} img/s | "
                    f"ETA {eta_s/60:.1f} min | "
                    f"GPU={_gpu_mem(device)}")
                t_last, n_last = now, end

        clip_arr = np.vstack(clip_parts).astype(np.float32)
        mnv3_arr = np.vstack(mnv3_parts).astype(np.float32)
        encode_s = time.perf_counter() - t0
        log(f"[GPU {rank}] {subj}: done  {n} imgs in {encode_s/60:.1f} min "
            f"({n/encode_s:.1f} img/s)")
        log(f"[GPU {rank}] {subj}: clip_arr={clip_arr.shape}  "
            f"mnv3_arr={mnv3_arr.shape}")

        # Copy bold_arr before closing shared memory (view into shm buffer)
        bold_copy = bold_arr.copy()
        shm_bold.close()
        shm_imgs.close()
        shm_offs.close()

        raw_path = os.path.join(out_dir, f"raw_{subj}.npz")
        np.savez_compressed(
            raw_path,
            bold_raw        = bold_copy,
            clip_embeds     = clip_arr,
            mobilenet_feats = mnv3_arr,        # ← NEW
            img_names       = np.array(meta["img_names"]),
            labels          = np.array(meta["labels"]),
            is_rep          = np.array(meta["is_rep"], dtype=bool),
        )
        npz_mb = os.path.getsize(raw_path) / 1e6
        log(f"[GPU {rank}] Saved raw_{subj}.npz  ({npz_mb:.0f} MB)")
        del clip_parts, mnv3_parts, clip_arr, mnv3_arr, bold_copy

    mnv3_model.remove_hooks()
    log(f"[GPU {rank}] Worker complete  GPU={_gpu_mem(device)}")
'''

_worker_path = "/kaggle/working/wave_worker.py"
with open(_worker_path, "w") as _f:
    _f.write(_WORKER_SRC)

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from wave_worker import encode_worker  # noqa
log(f"Worker written to {_worker_path} and imported ✓")

# ── CELL 10 : LAUNCH ENCODING WORKERS ─────────────────────────────────────────
import torch.multiprocessing as tmp

n_gpus = torch.cuda.device_count()
log(f"\nDetected {n_gpus} GPU(s)")
log("Launching encoding workers …\n")

if n_gpus >= 2:
    log("Spawning 2 parallel workers (one per GPU) …")
    tmp.spawn(encode_worker, args=(CFG, shm_meta), nprocs=2, join=True)
else:
    log("1 GPU — running workers sequentially on cuda:0")
    all_s = CFG["subjects"]
    mid   = len(all_s) // 2
    for rank, subj_slice in enumerate([all_s[:mid], all_s[mid:]]):
        _cfg_override = {**CFG, "subjects": subj_slice}
        encode_worker(0, _cfg_override, shm_meta)

log("\n✓ Encoding complete.")

# ── CELL 11 : RELEASE SHARED MEMORY ───────────────────────────────────────────
log("\nReleasing shared memory …")
for subj, (shm_bold, shm_imgs, shm_offs) in shm_handles.items():
    for shm in (shm_bold, shm_imgs, shm_offs):
        shm.close()
        shm.unlink()
    log(f"  {subj}: released")
del shm_handles, shm_meta

# ── CELL 12 : TEST SPLIT ──────────────────────────────────────────────────────
log("\nDetermining test split …")
rng_test      = random.Random(CFG["test_seed"])
all_rep_names = set()

for subj in SUBJECTS:
    raw_path = os.path.join(CFG["output_dir"], f"raw_{subj}.npz")
    d = np.load(raw_path, allow_pickle=True)
    for nm, rep in zip(d["img_names"], d["is_rep"]):
        if rep:
            all_rep_names.add(str(nm))

all_rep_names  = sorted(all_rep_names)
n_rep          = len(all_rep_names)
rng_test.shuffle(all_rep_names)
TEST_IMG_NAMES = set(all_rep_names[: CFG["n_test_hold_out"]])

log(f"  Unique repeated images : {n_rep}")
log(f"  Held-out test images   : {len(TEST_IMG_NAMES)}")

test_list_path = os.path.join(CFG["output_dir"], "test_image_names.json")
with open(test_list_path, "w") as f:
    json.dump(sorted(TEST_IMG_NAMES), f, indent=2)
log(f"  Saved → {test_list_path}")

# ── CELL 13 : NORMALISATION & SCORES ─────────────────────────────────────────
log("\nNormalising BOLD and computing scores …")

for subj in SUBJECTS:
    raw_path = os.path.join(CFG["output_dir"], f"raw_{subj}.npz")
    if not os.path.exists(raw_path):
        log(f"  {subj}: raw file missing — skipping")
        continue

    d               = np.load(raw_path, allow_pickle=True)
    bold_arr        = d["bold_raw"].astype(np.float32)
    clip_arr        = d["clip_embeds"].astype(np.float32)
    mobilenet_arr   = d["mobilenet_feats"].astype(np.float32)   # ← NEW
    img_names       = d["img_names"]
    labels          = d["labels"]
    is_rep          = d["is_rep"].astype(bool)
    n               = len(bold_arr)

    log(f"\n  {subj}: {n} samples")
    log(f"    clip shape        : {clip_arr.shape}")
    log(f"    mobilenet shape   : {mobilenet_arr.shape}")    # ← NEW

    is_test  = np.array([str(nm) in TEST_IMG_NAMES for nm in img_names], dtype=bool)
    is_train = ~is_test

    peak         = CFG["tr_peak_indices"]
    bold_targets = bold_arr[:, peak, :].mean(axis=1)

    train_bold   = bold_targets[is_train]
    parcel_mean  = train_bold.mean(axis=0, keepdims=True)
    parcel_std   = train_bold.std(axis=0,  keepdims=True) + 1e-8
    bold_zscored = (bold_targets - parcel_mean) / parcel_std

    limbic_raw   = bold_zscored[:, LIMBIC_MASK].mean(axis=1)
    train_limbic = limbic_raw[is_train]
    lmin, lmax   = train_limbic.min(), train_limbic.max()
    limbic_norm  = (limbic_raw - lmin) / (lmax - lmin + 1e-8)

    def net_mean(bz, mask):
        return bz[:, mask].mean(axis=1) if mask.sum() > 0 else np.zeros(len(bz))

    dmn_score    = net_mean(bold_zscored, DMN_MASK)
    visual_score = net_mean(bold_zscored, VISUAL_MASK)
    fpn_score    = net_mean(bold_zscored, FPN_MASK)

    log(f"    Train={is_train.sum()}  Test={is_test.sum()}  "
        f"Limbic parcels={LIMBIC_MASK.sum()}  "
        f"Limbic range=[{limbic_norm[is_train].min():.3f}, "
        f"{limbic_norm[is_train].max():.3f}]")

    out_path = os.path.join(CFG["output_dir"], f"subject_{subj}.npz")
    np.savez_compressed(
        out_path,
        clip_embeddings  = clip_arr,
        mobilenet_feats  = mobilenet_arr,    # ← NEW  shape (N, 184)
        bold_targets     = bold_zscored,
        limbic_score     = limbic_norm,
        dmn_score        = dmn_score,
        visual_score     = visual_score,
        fpn_score        = fpn_score,
        img_names        = img_names,
        labels           = labels,
        is_rep           = is_rep,
        is_test          = is_test,
        parcel_mean      = parcel_mean[0],
        parcel_std       = parcel_std[0],
        limbic_min       = np.array([lmin]),
        limbic_max       = np.array([lmax]),
        yeo_assignment   = yeo_assignment,
    )
    log(f"    Saved → {out_path}")

# ── CELL 14 : SANITY CHECK ────────────────────────────────────────────────────
log("\n── Sanity check ──────────────────────────────────────────────────────")
for subj in SUBJECTS:
    fpath = os.path.join(CFG["output_dir"], f"subject_{subj}.npz")
    if not os.path.exists(fpath):
        log(f"  {subj}: MISSING")
        continue
    d       = np.load(fpath, allow_pickle=True)
    n_train = (~d["is_test"]).sum()
    n_test  = d["is_test"].sum()
    log(f"  {subj}: total={len(d['clip_embeddings'])}  "
        f"train={n_train}  test={n_test}  "
        f"clips={d['clip_embeddings'].shape}  "
        f"mnv3={d['mobilenet_feats'].shape}  "      # ← NEW
        f"bolds={d['bold_targets'].shape}  "
        f"limbic_mean={d['limbic_score'][~d['is_test']].mean():.3f}")

log("\n✓ Preprocessing complete.")
log(f"Files in {CFG['output_dir']}:")
for f in sorted(Path(CFG["output_dir"]).iterdir()):
    mb = f.stat().st_size / 1e6
    log(f"  {f.name}  {mb:.1f} MB")